In [1]:
# ============================================================
# CELL 1 — Install
# ============================================================
!pip install -q openai-whisper transformers datasets accelerate peft bitsandbytes
!pip install -q soundfile pandas numpy scikit-learn

In [ ]:
# ============================================================
# CELL 2 — Imports & Config
# ============================================================
import os, gc, re, random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from huggingface_hub import login
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

HF_TOKEN = "****************"
login(token=HF_TOKEN)

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# ============================================================
# CELL 3 — Whisper Transcription
# ============================================================
import whisper

torch.cuda.empty_cache(); gc.collect()

train_cc_files, train_cd_files, test_audio_files = [], [], []
TEST_INFO_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        fp = os.path.join(root, f)
        if f == "testSetInfo.csv":           TEST_INFO_PATH = fp
        elif f.endswith(".wav"):
            rl = root.lower()
            if   "/cc" in rl: train_cc_files.append(fp)
            elif "/cd" in rl: train_cd_files.append(fp)
            elif "/audio" in rl or "test" in rl: test_audio_files.append(fp)

print(f"CC:{len(train_cc_files)} CD:{len(train_cd_files)} Test:{len(test_audio_files)}")

wmodel = whisper.load_model("large-v2", device=DEVICE)

def transcribe(file_list, label):
    out = []
    for f in tqdm(file_list, desc=f"label={label}"):
        pid = os.path.basename(f).replace('.wav','').strip()
        try:
            r = wmodel.transcribe(f)
            out.append({"id": pid, "text": r['text'].strip(), "label": label})
        except Exception as e:
            print(f"ERR {f}: {e}")
    return out

train_recs = transcribe(train_cc_files, 0) + transcribe(train_cd_files, 1)
test_recs  = transcribe(test_audio_files, -1)

df_train = pd.DataFrame(train_recs)
df_test  = pd.DataFrame(test_recs)

meta = pd.read_csv(TEST_INFO_PATH)
meta.columns = meta.columns.str.strip()
id_col    = meta.columns[0]
label_col = next((c for c in meta.columns if c.lower() in ['label','class','group']), meta.columns[1])

def parse_label(v):
    s = str(v).lower()
    return 1 if any(x in s for x in ['dem','ad','1']) else 0

lmap = dict(zip(meta[id_col].astype(str).str.strip(), meta[label_col].apply(parse_label)))
df_test['label'] = df_test['id'].map(lmap)

df_train.to_csv("train_whisper.csv", index=False)
df_test.to_csv("test_whisper.csv",   index=False)
print(f"Saved train={len(df_train)}, test={len(df_test)}")

del wmodel; torch.cuda.empty_cache(); gc.collect()
print("Whisper done.")

CC:54 CD:54 Test:48


label=-1: 100%|██████████| 48/48 [07:55<00:00,  9.91s/it]


Saved train=108, test=48
Whisper done.


In [4]:
# ============================================================
# CELL 4 — Richer Feature Extraction + CoT Prompt Builder
#
# IMPROVEMENT 1: Richer CoT prompt
# Beyond just cue proportion, we surface 3 additional
# linguistically-grounded AD biomarkers into the prompt:
#
#   a) Filler/hesitation words — AD patients use significantly
#      more fillers (uh, um, er, well, like, you know)
#   b) Repetition score — repeated words/phrases are a key
#      AD linguistic marker
#   c) Sentence count — AD patients produce shorter, more
#      fragmented descriptions
#
# These are all mentioned in the paper's Related Works (Section 2)
# as known AD biomarkers. Adding them to the prompt gives the
# model richer reasoning material without changing the architecture.
# ============================================================

CUE_LIST = ["stool","sink","dish","wash","jar","cookie",
            "child","mother","window","cabinet","kitchen","water"]

FILLER_WORDS = ["uh","um","er","erm","hmm","well","like",
                "you know","i mean","sort of","kind of"]

def extract_features(text):
    text_lower = str(text).lower()
    words = text_lower.split()
    n_words = max(len(words), 1)

    # 1. Cue proportion
    mentioned_cues = [c for c in CUE_LIST if c in text_lower]
    cue_prop = len(mentioned_cues) / len(CUE_LIST)

    # 2. Filler word rate (per 100 words)
    filler_count = sum(text_lower.count(fw) for fw in FILLER_WORDS)
    filler_rate  = round(filler_count / n_words * 100, 1)

    # 3. Repetition score — count words appearing more than twice
    from collections import Counter
    word_counts  = Counter(w for w in words if len(w) > 3)  # ignore short words
    repeated     = sum(1 for w, c in word_counts.items() if c > 2)
    repeat_score = round(repeated / max(len(word_counts), 1) * 100, 1)

    # 4. Sentence count (proxy for speech fragmentation)
    sentences    = re.split(r'[.!?]+', text.strip())
    n_sentences  = len([s for s in sentences if len(s.strip()) > 5])

    return {
        "cue_prop":      cue_prop,
        "mentioned_cues": mentioned_cues,
        "filler_rate":   filler_rate,
        "repeat_score":  repeat_score,
        "n_sentences":   n_sentences,
        "n_words":       n_words
    }

def build_cot_prompt(text, shuffle_steps=False):
    """Build the paper's CoT prompt + richer linguistic features."""
    f = extract_features(text)
    mentioned_str = ", ".join(f['mentioned_cues']) if f['mentioned_cues'] else "none"

    system_msg = (
        "You are an expert neurologist specializing in dementia diagnosis. "
        "Analyze the participant's description of the kitchen scene image "
        "for signs of cognitive impairment."
    )

    # Linguistic feature summary injected before analysis steps
    feature_summary = (
        f"Linguistic Profile:\n"
        f"- Scene elements mentioned: {len(f['mentioned_cues'])}/12 ({f['cue_prop']:.0%}): {mentioned_str}\n"
        f"- Hesitation/filler word rate: {f['filler_rate']} per 100 words\n"
        f"- Word repetition score: {f['repeat_score']}% (higher = more repetition)\n"
        f"- Sentence count: {f['n_sentences']} (fewer may indicate fragmented speech)\n"
        f"- Total word count: {f['n_words']}\n"
    )

    steps = [
        "1. Check for mention of key cue elements (e.g., stool, sink, dish, etc.).",
        "2. Evaluate awareness of safety hazards (e.g., stool, water, window).",
        "3. Assess logical flow and narrative coherence using connecting words.",
        "4. Integrate all findings to determine the likelihood of dementia."
    ]
    if shuffle_steps:  # for data augmentation variant
        mid = steps[1:3]
        random.shuffle(mid)
        steps = [steps[0]] + mid + [steps[3]]

    user_msg = (
        f"Transcript: {text}\n\n"
        f"{feature_summary}\n"
        f"Analysis Steps:\n"
        + "\n".join(steps)
        + "\n\nBased on the analysis above, classify the participant as: Dementia or Control."
    )

    return [{"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg}]

# Load transcriptions
df_train = pd.read_csv("train_whisper.csv").dropna(subset=['text'])
df_test  = pd.read_csv("test_whisper.csv").dropna(subset=['text'])

df_train['messages'] = df_train['text'].apply(lambda t: build_cot_prompt(t, shuffle_steps=False))
df_test['messages']  = df_test['text'].apply(lambda t: build_cot_prompt(t, shuffle_steps=False))

print("Prompts built. Sample:")
print(df_train['messages'].iloc[0][1]['content'][:600])

Prompts built. Sample:
Transcript: Put that picture and tell me everything that you see going on in that picture, everything that's happening. A lot of things are happening. Yes. The water's going over, the water's overflowing, and the little boy is slipping off the stool while he's trying to steal some cookies. And the little girl is laughing at him and she's not helping him, what he did. And the lady is drying the dishes and looking out the window. Beautiful out there, must be June. I don't know, but at any height, it looks like it's not winter. That's about all I can see. Okay, that's fine.

Linguistic Profile:
-


In [5]:
# ============================================================
# CELL 5 — Data Augmentation
#
# IMPROVEMENT 2: Double the training set
# Each sample gets a second variant with shuffled middle
# analysis steps (steps 2 & 3 swapped). This:
#   - Doubles training data: 108 → 216 samples
#   - Teaches the model the reasoning is step-order-agnostic
#   - Reduces overfitting on the tiny dataset
# Labels are preserved exactly.
# ============================================================

df_aug = df_train.copy()
df_aug['messages'] = df_aug['text'].apply(lambda t: build_cot_prompt(t, shuffle_steps=True))

df_train_full = pd.concat([df_train, df_aug], ignore_index=True)
print(f"Augmented training set: {len(df_train)} → {len(df_train_full)} samples")
print(f"Label distribution: {df_train_full['label'].value_counts().to_dict()}")

Augmented training set: 108 → 216 samples
Label distribution: {0: 108, 1: 108}


In [6]:
# ============================================================
# CELL 6 — Tokenizer
# ============================================================
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
MAX_LENGTH = 512

def tokenize(examples):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples['messages']
    ]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=MAX_LENGTH)

print("Tokenizer ready.")

Tokenizer ready.


In [7]:
# ============================================================
# CELL 7 — Train Single Model (call this per seed for ensemble)
#
# IMPROVEMENT 3: Calibrated hyperparameters for 108-sample regime
#   - lr = 3e-5 (more conservative than 1e-4)
#   - epochs = 6 with early stopping patience=2
#   - label_smoothing = 0.1
#   - lora_dropout = 0.05
# ============================================================
from transformers import (
    AutoModelForSequenceClassification, BitsAndBytesConfig,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

WARMUP_RATIO  = 0.1
SUSTAIN_RATIO = 0.3

def make_sustain_scheduler(optimizer, num_warmup_steps, num_sustain_steps, num_training_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return step / max(1, num_warmup_steps)
        elif step < num_warmup_steps + num_sustain_steps:
            return 1.0
        else:
            d = num_training_steps - num_warmup_steps - num_sustain_steps
            p = step - num_warmup_steps - num_sustain_steps
            return max(0.0, 1.0 - p / max(1, d))
    from torch.optim.lr_scheduler import LambdaLR
    return LambdaLR(optimizer, lr_lambda)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average='macro')}

def train_model(seed, df_train_full, df_test):
    set_seed(seed)
    print(f"\n{'='*50}")
    print(f"Training model with seed={seed}")
    print(f"{'='*50}")

    # Stratified val split from augmented training set
    df_tr, df_val = train_test_split(
        df_train_full, test_size=0.15, random_state=seed,
        stratify=df_train_full['label']
    )
    print(f"  Train: {len(df_tr)} | Val: {len(df_val)}")

    train_ds = Dataset.from_pandas(df_tr[['messages','label']].reset_index(drop=True))
    val_ds   = Dataset.from_pandas(df_val[['messages','label']].reset_index(drop=True))
    test_ds  = Dataset.from_pandas(df_test[['messages','label']].reset_index(drop=True))

    tok_train = train_ds.map(tokenize, batched=True, remove_columns=['messages'])
    tok_val   = val_ds.map(tokenize,   batched=True, remove_columns=['messages'])
    tok_test  = test_ds.map(tokenize,  batched=True, remove_columns=['messages'])

    # Load fresh model each time
    torch.cuda.empty_cache(); gc.collect()

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=2, quantization_config=bnb,
        device_map="auto", token=HF_TOKEN
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model = prepare_model_for_kbit_training(model)

    peft_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16, lora_alpha=16, lora_dropout=0.05,
        target_modules=["q_proj","v_proj","k_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        bias="none"
    )
    model = get_peft_model(model, peft_cfg)

    class SustainTrainer(Trainer):
        def create_scheduler(self, num_training_steps, optimizer=None):
            if optimizer is None: optimizer = self.optimizer
            nw = int(WARMUP_RATIO  * num_training_steps)
            ns = int(SUSTAIN_RATIO * num_training_steps)
            self.lr_scheduler = make_sustain_scheduler(optimizer, nw, ns, num_training_steps)
            return self.lr_scheduler

    args = TrainingArguments(
        output_dir=f"./run_seed{seed}",
        learning_rate=3e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=6,
        weight_decay=0.001,
        label_smoothing_factor=0.1,
        logging_steps=5,
        fp16=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        report_to="none",
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="linear",
    )

    trainer = SustainTrainer(
        model=model,
        args=args,
        train_dataset=tok_train,
        eval_dataset=tok_val,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    print(f"  Best val F1: {trainer.state.best_metric:.4f}")

    # Get raw logits for ensemble
    preds_out = trainer.predict(tok_test)
    logits    = preds_out.predictions  # shape (48, 2)

    # Free GPU memory before next seed
    del model, trainer
    torch.cuda.empty_cache(); gc.collect()

    return logits

print("Train function defined.")

2026-02-21 09:17:39.217941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771665459.242970    1543 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771665459.249375    1543 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771665459.266096    1543 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771665459.266118    1543 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771665459.266120    1543 computation_placer.cc:177] computation placer alr

Train function defined.


In [8]:
# ============================================================
# CELL 8 — Ensemble over 3 Seeds
#
# IMPROVEMENT 4: Ensemble
# On a 48-sample test set, each correct prediction = +2.08%.
# Variance across runs is high. Ensembling 3 seeds with
# averaged logits (soft voting) smooths this out and
# consistently beats any single run.
#
# Seeds chosen to be diverse: 42, 7, 123
# ============================================================

SEEDS = [42, 7, 123]
all_logits = []

for seed in SEEDS:
    logits = train_model(seed, df_train_full, df_test)
    all_logits.append(logits)
    print(f"Seed {seed} done. Individual accuracy: "
          f"{accuracy_score(df_test['label'].values, np.argmax(logits, axis=-1))*100:.2f}%")

# Soft voting: average logits across all 3 models
ensemble_logits = np.mean(all_logits, axis=0)
ensemble_preds  = np.argmax(ensemble_logits, axis=-1)
true_labels     = df_test['label'].values

acc = accuracy_score(true_labels, ensemble_preds) * 100
f1  = f1_score(true_labels, ensemble_preds, average='macro') * 100

print("\n" + "="*55)
print(f"  ENSEMBLE RESULT (3 seeds, soft voting)")
print(f"  Accuracy : {acc:.2f}%  (Paper target: 83.33%)")
print(f"  F1 Score : {f1:.2f}%  (Paper target: 83.22%)")
print("="*55)
print(classification_report(true_labels, ensemble_preds, target_names=['Control','Dementia']))


Training model with seed=42
  Train: 183 | Val: 33


Map:   0%|          | 0/183 [00:00<?, ? examples/s]

Map:   0%|          | 0/33 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.287600,0.706910,0.545455,0.476190
2,1.016100,0.628663,0.575758,0.465278
3,0.704600,0.549170,0.818182,0.809615
4,0.608500,0.515799,0.818182,0.813910
5,0.556400,0.468668,0.757576,0.757353
6,0.422200,0.438694,0.848485,0.846226


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

  Best val F1: 0.8462


Seed 123 done. Individual accuracy: 75.00%

  ENSEMBLE RESULT (3 seeds, soft voting)
  Accuracy : 81.25%  (Paper target: 83.33%)
  F1 Score : 81.24%  (Paper target: 83.22%)
              precision    recall  f1-score   support

     Control       0.80      0.83      0.82        24
    Dementia       0.83      0.79      0.81        24

    accuracy                           0.81        48
   macro avg       0.81      0.81      0.81        48
weighted avg       0.81      0.81      0.81        48



In [9]:
# ============================================================
# CELL 9 — Per-Seed Breakdown
# Shows individual model performance so you can see
# how much the ensemble helps vs individual models
# ============================================================
print("Per-seed individual results:")
print("-" * 40)
for seed, logits in zip(SEEDS, all_logits):
    preds = np.argmax(logits, axis=-1)
    a = accuracy_score(true_labels, preds) * 100
    f = f1_score(true_labels, preds, average='macro') * 100
    print(f"  Seed {seed:3d}: Acc={a:.2f}%  F1={f:.2f}%")
print("-" * 40)
print(f"  Ensemble: Acc={acc:.2f}%  F1={f1:.2f}%")

Per-seed individual results:
----------------------------------------
  Seed  42: Acc=72.92%  F1=72.90%
  Seed   7: Acc=66.67%  F1=66.61%
  Seed 123: Acc=75.00%  F1=74.83%
----------------------------------------
  Ensemble: Acc=81.25%  F1=81.24%
